In [ ]:
SEQUENCE TO SEQUENCE
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences



print("Enter training text.")
print("Type your paragraph and press ENTER when finished.\n")

training_text = input("Training paragraph: ")


training_text = training_text.lower()


if len(training_text.strip()) == 0:
    print("No text provided.")
    exit()



tokenizer = Tokenizer()

tokenizer.fit_on_texts([training_text])

total_words = len(tokenizer.word_index) + 1


token_list = tokenizer.texts_to_sequences(
    [training_text]
)[0]


input_sequences = []

for i in range(1, len(token_list)):
    sequence = token_list[:i + 1]
    input_sequences.append(sequence)


if len(input_sequences) < 2:
    print("Please enter a longer paragraph.")
    exit()


max_sequence_len = max(
    len(sequence)
    for sequence in input_sequences
)

input_sequences = np.array(
    pad_sequences(
        input_sequences,
        maxlen=max_sequence_len,
        padding='pre'
    )
)


X = input_sequences[:, :-1]
y = input_sequences[:, -1]



model = Sequential([
    Embedding(
        total_words,
        32,
        input_length=max_sequence_len - 1
    ),

    SimpleRNN(
        64,
        activation='tanh'
    ),

    Dense(
        total_words,
        activation='softmax'
    )
])


model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)



model.fit(
    X,
    y,
    epochs=100,
    verbose=0
)

print("\nRNN model trained successfully!")



user_text = input(
    "\nEnter a sentence/paragraph to predict the next word: "
)

user_text = user_text.lower()


token_list = tokenizer.texts_to_sequences(
    [user_text]
)[0]

if len(token_list) == 0:

    print(
        "\nThe entered words were not found "
        "in the training text."
    )

else:


    token_list = pad_sequences(
        [token_list],
        maxlen=max_sequence_len - 1,
        padding='pre'
    )


    prediction = model.predict(
        token_list,
        verbose=0
    )

    predicted_index = np.argmax(
        prediction,
        axis=1
    )[0]


    predicted_word = None

    for word, index in tokenizer.word_index.items():

        if index == predicted_index:
            predicted_word = word
            break

    print("\nInput:")
    print(user_text)

    print("\nPredicted next word:")
    print(predicted_word)

TEMPORAL DATA:

import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense

print("==============================================")
print("   RNN TEMPORAL DATA CLASSIFICATION")
print("==============================================")

n_samples = int(
    input("\nEnter number of training samples: ")
)

time_steps = int(
    input("Enter number of values in each sequence: ")
)

X = []
y = []

for i in range(n_samples):

    print(f"\nTraining sample {i + 1}")

    while True:
        values = input(
            "Enter sequence: "
        ).split()

        if len(values) != time_steps:
            print(
                f"Please enter exactly {time_steps} values."
            )
        else:
            break

    values = [float(value) for value in values]
    X.append(values)

    while True:
        label = int(
            input("Enter class (0 or 1): ")
        )

        if label in [0, 1]:
            break

        print("Class must be either 0 or 1.")

    y.append(label)

X = np.array(X, dtype=float)
y = np.array(y, dtype=int)

X = X.reshape(
    n_samples,
    time_steps,
    1
)

model = Sequential([
    SimpleRNN(
        32,
        activation='tanh',
        input_shape=(time_steps, 1)
    ),
    Dense(
        16,
        activation='relu'
    ),
    Dense(
        1,
        activation='sigmoid'
    )
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("\nTraining RNN...")

model.fit(
    X,
    y,
    epochs=100,
    verbose=0
)

print("\nRNN model trained successfully!")

while True:

    user_input = input(
        f"\nEnter {time_steps} values to predict: "
    ).split()

    if len(user_input) != time_steps:
        print(
            f"Please enter exactly {time_steps} values."
        )
    else:
        break

values = [
    float(value)
    for value in user_input
]

test_data = np.array(
    values,
    dtype=float
).reshape(
    1,
    time_steps,
    1
)

prediction = model.predict(
    test_data,
    verbose=0
)[0][0]

if prediction >= 0.5:
    result = 1
else:
    result = 0

print("\n==============================================")
print("             FINAL RESULT")
print("==============================================")

print("\nInput sequence:")
print(values)

print("\nProbability of Class 1:")
print(round(float(prediction), 4))

print("\nProbability of Class 0:")
print(round(float(1 - prediction), 4))

print("\nPredicted Class:")
print(result)

print("==============================================")